#### Definição das variáveis de interesse da separação por perfil de município

1. Eliminação de Multicolinearidade

2. Análise de Componentes Principais (PCA)

3. Selecionar os mais fortes no ranking da PCA 

In [7]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent
file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"

df = pd.read_csv(file_path)

df.head()

,id_municipio,População_2022,Pontuação Indice ODS 2025,Goal 1 Score,Goal 2 Score,Goal 3 Score,Goal 5 Score,Goal 6 Score,Goal 7 Score,Goal 8 Score,...,Normalizado (0-100): SDG15_8_UNI_CNS,Normalizado (0-100): SDG15_PRTAMB,Normalizado (0-100): SDG16_1_OB_HM,Normalizado (0-100): SDG16_3_M_ARM_FG,Normalizado (0-100): SDG16_4_TX_HMC,Normalizado (0-100): SDG16_CRP,Normalizado (0-100): SDG16_DRTHMN,Normalizado (0-100): SDG16_TRANSP,Normalizado (0-100): SDG17_1_INVST_PB,Normalizado (0-100): SDG17_3_P_RC_TRB
0,1100015,21495,41.885064,53.769451,53.255278,66.451721,58.730307,20.177142,56.309444,46.940699,...,30.659,80,0.0,0.0,38.784211,85.714,42.857,66.667,0.000000,8.108054
1,1100023,96833,45.475390,54.219225,50.890756,65.677811,41.654565,72.408554,68.007222,49.281375,...,0.013,80,0.0,0.0,32.057895,85.714,57.143,66.667,1.133196,34.298246
2,1100031,5363,47.569518,49.228293,43.817398,89.494428,48.938353,44.025511,63.241944,54.613778,...,0.000,0,0.0,0.0,1.642105,85.714,42.857,66.667,38.715329,8.963317
3,1100049,86895,48.582323,50.752090,51.083977,73.026011,47.023041,66.630268,69.962222,45.581682,...,0.012,80,0.0,0.0,63.655263,85.714,57.143,50.000,12.187507,40.743620
4,1100056,15890,48.718897,56.552994,50.268036,79.958615,32.086768,65.794541,68.762222,52.890151,...,40.055,80,100.0,0.0,33.755263,100.000,42.857,66.667,8.461212,21.905901


Filtro Estrito para os Municípios de Santa Catarina (SC)

In [8]:
raw_path = base_path / "data" / "raw" / "Base_de_Dados_IDSC-BR_2025.xlsx"
uf_por_municipio = pd.read_excel(raw_path, sheet_name="Todos os Dados", usecols=["ID", "UF"])
uf_por_municipio = uf_por_municipio.rename(columns={"ID": "id_municipio"})

ids_sc = uf_por_municipio.loc[uf_por_municipio["UF"] == "SC", "id_municipio"]
df = df[df["id_municipio"].isin(ids_sc)].reset_index(drop=True)

print(f"Municípios de Santa Catarina na base: {df.shape[0]}")

Municípios de Santa Catarina na base: 295


Divisão Treino e Teste (Específica para SC)

Calcula a matriz de correlação e descarta variáveis redundantes priorizando a correlação média com o restante da base.

In [9]:
import pandas as pd
import numpy as np

df_model = df

corr_matrix = df_model.corr(numeric_only=True).abs()
mean_corr = corr_matrix.mean()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = set()
pairs_above_threshold = []

for column in upper.columns:
    correlated = upper.index[upper[column] > 0.80].tolist()
    for row in correlated:
        pairs_above_threshold.append((column, row))
        loser = column if mean_corr[column] >= mean_corr[row] else row
        to_drop.add(loser)

X_filtered = df_model.drop(columns=list(to_drop), errors='ignore')

print(f"Total de variáveis iniciais: {len(df_model.columns)}")
print(f"Pares com correlação > 0.80 encontrados: {len(pairs_above_threshold)}")
print(f"Total após remover redundantes (Correlação > 0.8): {len(X_filtered.columns)}")

Total de variáveis iniciais: 135
Pares com correlação > 0.80 encontrados: 15
Total após remover redundantes (Correlação > 0.8): 121


Remoção de Targets Educacionais e Imputação de Nulos

Remove variáveis ligadas ao IDEB para evitar vazamento de dados e preenche valores ausentes com a mediana do treino.

In [10]:
# Remove colunas do SDG4 (IDEB e educação) para não vazar a resposta nas features
cols_sdg4 = [c for c in X_filtered.columns if 'SDG4' in c or 'ideb' in c.lower()]
X_filtered = X_filtered.drop(columns=cols_sdg4, errors='ignore')

# Imputação de valores nulos com a mediana
medianas_treino = X_filtered.median(numeric_only=True)
X_filtered = X_filtered.fillna(medianas_treino)

Aplicação da Análise de Componentes Principais (PCA)

Padroniza os dados e aplica o PCA para extração de componentes ortogonais.

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# Definimos um número adequado de componentes para a base reduzida de SC
pca = PCA(n_components=15) 
pca.fit(X_scaled)

pc_columns = [f'PC{i}' for i in range(1, pca.n_components_ + 1)]
loadings = pd.DataFrame(pca.components_.T, columns=pc_columns, index=X_filtered.columns)

print(f"Variância explicada acumulada (15 componentes): {sum(pca.explained_variance_ratio_) * 100:.2f}%")

Variância explicada acumulada (15 componentes): 51.53%


Seleção das Principais Features via Loadings do PCA

Identifica e seleciona as variáveis originais de maior peso absoluto nos componentes principais.

In [13]:
import json

dir_processed = base_path / "data" / "processed"

# Garante que final_vars pegue todas as colunas remanescentes após o filtro (exceto o id e alvos se houverem)
final_vars = [col for col in X_filtered.columns if col != 'id_municipio']

# Salvamos a lista de features selecionadas para ser usada na classificação
with open(dir_processed / "features_selecionadas_sc.json", "w", encoding="utf-8") as f:
    json.dump(final_vars, f, ensure_ascii=False, indent=2)

print(f"{len(final_vars)} variáveis selecionadas salvas com sucesso em -> {dir_processed / 'features_selecionadas_sc.json'}")

105 variáveis selecionadas salvas com sucesso em -> /home/clara/Desktop/idsc-analysis/data/processed/features_selecionadas_sc.json
